In [3]:
from okx_plug import InitOkxAdaptor
from upbit_plug import InitUpbitAdaptor
from binance_plug import InitBinanceAdaptor
from bithumb_plug import InitBithumbAdaptor
from bybit_plug import InitBybitAdaptor
import pandas as pd
import json
import time
import datetime
from threading import Thread
import numpy as np

import sys
import os
import datetime
import traceback
import pandas as pd
import numpy as np
import time
from binance.client import Client

sys.path.append('/home/trade_core')
from loggers.logger import KimpBotLogger

In [4]:
binance_access_key = '4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb'
binance_secret_key = '011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK'

upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

okx_access_key = "fcb66a6e-0443-4743-8d9b-61caf7eab662"
okx_secret_key = "0029E09874B38F5AC7E68E9DFC348667"
okx_passphrase = "121431Rn!!"

bybit_access_key = "S3YKBbD0kz1WwcfqZD"
bybit_secret_key = "390M6dKJ9J0uEK7vXYAl3qxVCh944tRrzHah"

In [5]:
class UserBinanceAdaptor:
    def __init__(self, recvWindow=45000, logging_dir=None):
        self.user_client_dict = {}
        self.recvWindow = recvWindow
        self.logger = KimpBotLogger("user_binance_plug", logging_dir).logger
        self.retry_term_sec = 0.2
        self.retry_count_limit = 2
        self.logger.info(f"user_binance_plug_logger started.")

    def load_user_client(self, access_key, secret_key):
        user_client = self.user_client_dict.get(access_key)
        if user_client is None:
            self.user_client_dict[access_key] = Client(access_key, secret_key)
            return self.user_client_dict[access_key]
        else:
            return user_client

    def check_api_key(self, access_key, secret_key, futures=False):
        self.user_client_dict.pop(access_key, None)
        try:
            if futures is False:
                self.get_spot_balance(access_key, secret_key)
            else:
                self.get_usdm_balance(access_key, secret_key)
            return (True, 'OK')
        except Exception as e:
            self.user_client_dict.pop(access_key, None)
            return (False, str(e))

    def get_spot_balance(self, access_key, secret_key):
        client = self.load_user_client(access_key, secret_key)
        spot_balance = pd.DataFrame(client.get_account()['balances'])
        spot_balance.loc[:,['free','locked']] = spot_balance[['free','locked']].astype(float)
        spot_balance_df = spot_balance[spot_balance['free']>0].reset_index(drop=True)
        return spot_balance_df

    def get_usdm_balance(self, access_key, secret_key, return_dict=None):
        if return_dict is None:
            client = self.load_user_client(access_key, secret_key)
            usdm_balance_df = pd.DataFrame(client.futures_account_balance())
            usdm_balance_df.loc[:, ['balance', 'maxWithdrawAmount']] = usdm_balance_df[['balance', 'maxWithdrawAmount']].astype(float)
            return usdm_balance_df
        else:
            client = self.load_user_client(access_key, secret_key)
            usdm_balance_df = pd.DataFrame(client.futures_account_balance())
            usdm_balance_df.loc[:, ['balance', 'maxWithdrawAmount']] = usdm_balance_df[['balance', 'maxWithdrawAmount']].astype(float)
            return_dict['res'] = usdm_balance_df

    def get_balance(self, access_key, secret_key, market_type='SPOT'):
        if market_type == "SPOT":
            return self.get_spot_balance(access_key, secret_key)
        elif market_type == "USD_M":
            return self.get_usdm_balance(access_key, secret_key)
        elif market_type == "COIN_M":
            raise Exception(f"market_type: {market_type} is not supported yet.")
        else:
            raise Exception(f"Invalid market_type: {market_type}")

    # Binancef order functions
    def change_margin_type(self, access_key, secret_key, symbol, marginType='ISOLATED'):
        client = self.load_user_client(access_key, secret_key)
        try:
            res = client.futures_change_margin_type(
                symbol=symbol,
                marginType=marginType
            )
        except Exception as e:
            error_code = e.code
            if error_code == -4046:
                res = {'code': 200, 'msg': e.message}
            else:
                self.logger.error(f"change_margin_type|{traceback.format_exc()}")
                raise Exception(e)
        return res

    def change_leverage(self, access_key, secret_key, symbol, leverage):
        client = self.load_user_client(access_key, secret_key)
        res = client.futures_change_leverage(symbol=symbol, leverage=leverage)
        return res

    def futures_order_info(self, access_key, secret_key, symbol, orderId, return_dict=None):
        client = self.load_user_client(access_key, secret_key)
        if return_dict is None:
            res = client.futures_get_order(symbol=symbol, orderId=orderId, recvWindow=self.recvWindow)
            return res
        else:
            try:
                return_dict['res'] = client.futures_get_order(symbol=symbol, orderId=orderId, recvWindow=self.recvWindow)
                return_dict['state'] = 'OK'
            except Exception as e:
                self.logger.error(f"futures_order_info|{traceback.format_exc()}")
                return_dict['res'] = e
                return_dict['state'] = 'ERROR'
    # Original
    # def market_enter(self, access_key, secret_key, symbol, qty, return_dict=None):
    #     client = self.load_user_client(access_key, secret_key)
    #     if return_dict is None:
    #         res = client.futures_create_order(
    #             symbol=symbol,
    #             side='SELL',
    #             type='MARKET',
    #             quantity=qty,
    #             recvWindow=self.recvWindow
    #             # workingType='MARK_PRICE'
    #         )
    #         return res
    #     else:
    #         try:
    #             return_dict['res'] =  client.futures_create_order(
    #                 symbol=symbol,
    #                 side='SELL',
    #                 type='MARKET',
    #                 quantity=qty,
    #                 recvWindow=self.recvWindow
    #                 # workingType='MARK_PRICE'
    #             )
    #             return_dict['state'] = 'OK'
    #         except Exception as e:
    #             self.logger.error(f"market_enter|{traceback.format_exc()}")
    #             return_dict['res'] = e
    #             return_dict['state'] = 'ERROR'

    def market_enter(self, access_key, secret_key, symbol, qty, return_dict=None):
        client = self.load_user_client(access_key, secret_key)

        retry_count = 0

        while retry_count <= self.retry_count_limit:
            if retry_count >= 1:
                self.logger.info(f"market_enter|retry_count: {retry_count}, retry_term_sec: {self.retry_term_sec}, retry_count_limit: {self.retry_count_limit}") # For TESTING
            try:
                res = client.futures_create_order(
                    symbol=symbol,
                    side='SELL',
                    type='MARKET',
                    quantity=qty,
                    recvWindow=self.recvWindow
                )
                if return_dict is not None:
                    return_dict['res'] = res
                    return_dict['state'] = 'OK'
                return res
            except Exception as e:
                if e.code not in [-1000, -1001] or retry_count == self.retry_count_limit:
                    if return_dict is None:
                        raise e
                    else:
                        return_dict['res'] = e
                        return_dict['state'] = 'ERROR'
                        return
            retry_count += 1

    # # Original
    # def market_exit(self, access_key, secret_key, symbol, qty, return_dict=None):
    #     client = self.load_user_client(access_key, secret_key)
    #     if return_dict is None:
    #         res = client.futures_create_order(
    #             symbol=symbol,
    #             side='BUY',
    #             type='MARKET',
    #             quantity=qty,
    #             recvWindow=self.recvWindow
    #         )
    #         return res
    #     else:
    #         try:
    #             return_dict['res'] = client.futures_create_order(
    #             symbol=symbol,
    #             side='BUY',
    #             type='MARKET',
    #             quantity=qty,
    #             recvWindow=self.recvWindow
    #         )
    #             return_dict['state'] = 'OK'
    #         except Exception as e:
    #             self.logger.error(f"market_exit|{traceback.format_exc()}")
    #             return_dict['res'] = e
    #             return_dict['state'] = 'ERROR'

    def market_exit(self, access_key, secret_key, symbol, qty, return_dict=None):
        client = self.load_user_client(access_key, secret_key)

        retry_count = 0

        while retry_count <= self.retry_count_limit:
            if retry_count >= 1:
                self.logger.info(f"market_exit|retry_count: {retry_count}, retry_term_sec: {self.retry_term_sec}, retry_count_limit: {self.retry_count_limit}") # For TESTING
            try:
                res = client.futures_create_order(
                symbol=symbol,
                side='BUY',
                type='MARKET',
                quantity=qty,
                recvWindow=self.recvWindow
                )
                if return_dict is not None:
                    return_dict['res'] = res
                    return_dict['state'] = 'OK'
                return res
            except Exception as e:
                if e.code not in [-1000, -1001] or retry_count == self.retry_count_limit:
                    if return_dict is None:
                        raise e
                    else:
                        return_dict['res'] = e
                        return_dict['state'] = 'ERROR'
                        return
            retry_count += 1
    # Original
    # def position_information(self, access_key, secret_key, symbol, return_dict=None):
    #     client = self.load_user_client(access_key, secret_key)
    #     if return_dict is None:
    #         res = client.futures_position_information()
    #         position_df = pd.DataFrame(res)
    #         symbol_df = position_df[position_df['symbol']==symbol]
    #         if len(symbol_df) == 0:
    #             position_qty = 0
    #             liquidation_price = None
    #         else:
    #             position_qty = abs(float(symbol_df['positionAmt'].iloc[0]))
    #             liquidation_price = float(symbol_df['liquidationPrice'].iloc[0])
    #         return (position_qty, liquidation_price)
    #     else:
    #         try:
    #             res = client.futures_position_information()
    #             position_df = pd.DataFrame(res)
    #             symbol_df = position_df[position_df['symbol']==symbol]
    #             if len(symbol_df) == 0:
    #                 position_qty = 0
    #                 liquidation_price = None
    #             else:
    #                 position_qty = abs(float(symbol_df['positionAmt'].iloc[0]))
    #                 liquidation_price = float(symbol_df['liquidationPrice'].iloc[0])
    #             return_dict['res'] = (position_qty, liquidation_price)
    #             return_dict['state'] = 'OK'
    #         except Exception as e:
    #             self.logger.error(f"position_information|{traceback.format_exc()}")
    #             return_dict['res'] = e
    #             return_dict['state'] = 'ERROR'

    def position_information(self, access_key, secret_key, symbol, return_dict=None):
        client = self.load_user_client(access_key, secret_key)

        retry_count = 0

        while retry_count <= self.retry_count_limit:
            try:
                res = client.futures_position_information()
                position_df = pd.DataFrame(res)
                symbol_df = position_df[position_df['symbol']==symbol]
                if len(symbol_df) == 0:
                    position_qty = 0
                    liquidation_price = None
                else:
                    position_qty = abs(float(symbol_df['positionAmt'].iloc[0]))
                    liquidation_price = float(symbol_df['liquidationPrice'].iloc[0])
                if return_dict is not None:
                    return_dict['res'] = (position_qty, liquidation_price)
                    return_dict['state'] = 'OK'
                return (position_qty, liquidation_price)
            except Exception as e:
                if e.code not in [-1000, -1001] or retry_count == self.retry_count_limit:
                    if return_dict is None:
                        raise e
                    else:
                        return_dict['res'] = e
                        return_dict['state'] = 'ERROR'
                        return
            retry_count += 1

    def all_position_information(self, access_key, secret_key, return_dict=None):
        client = self.load_user_client(access_key, secret_key)
        if return_dict is None:
            res = client.futures_position_information()
            position_df = pd.DataFrame(res)
            position_df.loc[:, 'positionAmt':'maxNotionalValue'] = position_df.loc[:,'positionAmt':'maxNotionalValue'].astype(float)
            position_df = position_df[position_df['positionAmt']!=0].reset_index(drop=True)
            return position_df
        else:
            try:
                res = client.futures_position_information()
                position_df = pd.DataFrame(res)
                position_df.loc[:, 'positionAmt':'maxNotionalValue'] = position_df.loc[:,'positionAmt':'maxNotionalValue'].astype(float)
                position_df = position_df[position_df['positionAmt']!=0].reset_index(drop=True)
                return_dict['res'] = position_df
                return_dict['state'] = 'OK'
            except Exception as e:
                self.logger.error(f"all_position_information|{traceback.format_exc()}")
                return_dict['res'] = e
                return_dict['state'] = 'ERROR'

    def get_futures_account(self, access_key, secret_key, return_dict=None):
        client = self.load_user_client(access_key, secret_key)
        if return_dict is None:
            res = client.futures_account()
            res_df = pd.DataFrame(res['assets'])
            res_df.loc[:, 'walletBalance':'updateTime'] = res_df.loc[:, 'walletBalance':'updateTime'].astype(float)
            return res_df
        else:
            try:
                res = client.futures_account()
                res_df = pd.DataFrame(res['assets'])
                res_df.loc[:, 'walletBalance':'updateTime'] = res_df.loc[:, 'walletBalance':'updateTime'].astype(float)
                return_dict['res'] = res_df
                return_dict['state'] = 'OK'
            except Exception as e:
                self.logger.error(f"get_futures_account|{traceback.format_exc()}")
                return_dict['res'] = e
                return_dict['state'] = 'ERROR'

In [6]:
user_binance_adaptor = UserBinanceAdaptor()

2024-02-17 09:34:24,510 - user_binance_plug - INFO - user_binance_plug_logger started.


In [7]:
client = user_binance_adaptor.load_user_client(binance_access_key, binance_secret_key)

In [21]:
temp = pd.DataFrame(client.get_all_coins_info())
temp[temp['coin']=='XRP']

,coin,depositAllEnable,withdrawAllEnable,name,free,locked,freeze,withdrawing,ipoing,ipoable,storage,isLegalMoney,trading,networkList
80,XRP,True,True,Ripple,0.86460033,0,0,0,0,0,0,False,True,"[{'network': 'BSC', 'coin': 'XRP', 'withdrawIn..."


In [23]:
user_binance_adaptor.all_position_information(binance_access_key, binance_secret_key)

,symbol,positionAmt,entryPrice,breakEvenPrice,markPrice,unRealizedProfit,liquidationPrice,leverage,maxNotionalValue,marginType,isolatedMargin,isAutoAddMargin,positionSide,notional,isolatedWallet,updateTime,isolated,adlQuantile
0,ETCUSDT,-291.78,24.797714,22.401123,26.397,-466.639765,120.8221,4.0,10000000.0,cross,0.00000000,false,BOTH,-7702.11666000,0,1708100728008,False,1
1,STXUSDT,-4944.0,1.464272,1.46354,2.561888,-5426.608956,8.051207,4.0,2400000.0,cross,0.00000000,false,BOTH,-12665.97194832,0,1705623903014,False,1
2,SOLUSDT,-42.0,84.171,84.128914,109.659412,-1070.513319,767.899165,4.0,20000000.0,cross,0.00000000,false,BOTH,-4605.69531870,0,1706055101469,False,1
3,GMTUSDT,-15175.0,0.2373,0.237181,0.272967,-541.2402,2.079001,4.0,2000000.0,cross,0.00000000,false,BOTH,-4142.26769975,0,1707269040692,False,1
4,XRPUSDT,-10227.0,0.5634,0.563118,0.5544,92.043,3.249169,4.0,12000000.0,cross,0.00000000,false,BOTH,-5669.84880000,0,1708100262308,False,1
5,SEIUSDT,-18497.0,0.851325,0.730756,0.9542,-1902.887754,2.395949,4.0,1200000.0,cross,0.00000000,false,BOTH,-17649.83740000,0,1708099217904,False,1
6,LSKUSDT,-2442.0,1.467641,1.466907,1.36221,257.464114,12.428958,4.0,500000.0,cross,0.00000000,false,BOTH,-3326.51596530,0,1707209882261,False,3
7,VETUSDT,-151592.0,0.047553,0.04753,0.044733,427.486408,0.22552,4.0,5000000.0,cross,0.00000000,false,BOTH,-6781.21041360,0,1708100362647,False,0
